In [1]:
import os
import sys
sys.path.insert(
    0, os.path.abspath('../../')
)

import json
import yaml

from pathlib import Path
from rich.console import Console
from rich.table import Table

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from softprompt.algorithms.sklearn.linear_model import (SoftLogisticRegression,
                                                        SoftLogisticRegressionCV)

from softprompt.metrics.evaluator import BinaryEvaluator

In [2]:
root_dir = Path("../../").resolve()
print("Root directory:", root_dir)

Root directory: /home/hgkahng/Workspaces/soft-prompt


## 1. Load oracle data

In [3]:
from typing import Union, Tuple

def load_oracle_imdb_data(directory: Union[str, Path]) -> Tuple[np.ndarray, np.ndarray]:
    
    _directory = Path(directory).resolve()
    X_train = np.load(_directory / "train.features.npy")
    y_train = np.load(_directory / "train.labels.npy")
    X_test = np.load(_directory / "test.features.npy")
    y_test = np.load(_directory / "test.labels.npy")
    
    return (X_train, y_train, X_test, y_test)

In [4]:
emb_save_dir = root_dir / "data/imdb/embeddings/openai/text-embedding-3-small"
X_train, y_train, X_test, y_test = load_oracle_imdb_data(emb_save_dir)

In [5]:
X_train.shape, y_train.shape

((25000, 1536), (25000,))

In [6]:
X_test.shape, y_test.shape

((25000, 1536), (25000,))

## 2. Load synthetic data
- Either `hard` or `soft`

In [ ]:
load_dir = root_dir / "results/imdb/2025-05-06_16:23:58"  # hard
load_dir = root_dir / "results/imdb/2025-05-06_17:36:19"  # soft
load_dir = root_dir / "results/imdb/2025-05-08_09:47:42"  # hard + CoT
load_dir = root_dir / "results/imdb/2025-05-08_02:41:02"  # soft + CoT

print("Model directory:", load_dir)
print(*os.listdir(load_dir), sep="\n")

Model directory: /home/hgkahng/Workspaces/soft-prompt/results/imdb/2025-05-06_17:36:19
template.jsonl
embeddings
config.yaml
data.jsonl
template_formatted.txt


In [114]:
# Print configurations

with open(load_dir / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

style = "red" if cfg['hard'] else "green"

table = Table(title="Configuration(s)")
table.add_column("Name", justify="right", style="white", no_wrap=True)
table.add_column("Value", justify="left", style=style)
_ = [table.add_row(k, str(v)) for k, v in cfg.items()]

console = Console()
console.print(table);

         Configuration(s)          
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃         Name ┃ Value            ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│   batch_size │ 50               │
│          cot │ False            │
│         data │ imdb             │
│         hard │ False            │
│ log_interval │ 2                │
│   max_tokens │ 16384            │
│        model │ gemini-2.0-flash │
│ num_examples │ None             │
│  sample_size │ 75000            │
│  temperature │ 1.0              │
└──────────────┴──────────────────┘

In [103]:
with open(load_dir / 'template_formatted.txt') as f:
    template_example = "".join(f.readlines())

print(template_example)

system:
You are tasked with generating realistic movie reviews to train a sentiment classifier.
Use a sentiment scale from 0 (negative) to 1 (positive).
Reviews should generally be between 100 and 500 words, avoiding overly short or excessively long responses.


human:
Generate a movie review with a sentiment score of 0.767.



In [104]:
with open(load_dir / "data.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

print(len(data))

76487


In [105]:
data[0]

{'label': 0.151,
 'text': "This movie was a complete disaster. From the terrible acting to the nonsensical plot, there was nothing to enjoy. The special effects looked like they were from a decade ago, and the dialogue was painful to listen to. I struggled to stay awake through the whole thing, and I left the theater feeling like I had wasted my time and money. I would not recommend this movie to anyone, not even my worst enemy. It's a cinematic black hole that sucks the joy out of life. The direction was amateurish, the script was poorly written, and the editing was choppy and confusing. The soundtrack was also completely inappropriate for the scenes it accompanied, adding to the overall sense of incoherence. I can't believe this movie even got made. It's a true testament to the depths of Hollywood's creative bankruptcy. Avoid this film at all costs; you'll thank me later. Seriously, go watch paint dry instead; it would be a more entertaining experience. I'm giving it a generous 0.5 s

In [106]:
labels = [d['label'] for d in data]
labels = np.array(labels)

hard_labels = (labels > 0.5).astype(int)
soft_labels = labels.copy()

print("Hard labels, shape:", hard_labels.shape)
print("Soft labels, shape:", soft_labels.shape)

Hard labels, shape: (76487,)
Soft labels, shape: (76487,)


In [107]:
embeddings = np.load(
    load_dir / "embeddings/openai/text-embedding-3-small/data.npy"
)
print(embeddings.shape)

(76487, 1536)


In [108]:
assert len(hard_labels) == embeddings.shape[0]
assert len(soft_labels) == embeddings.shape[0]

In [109]:
# aliasing for readabillity
X_syn = embeddings.copy()
y_syn_hard = hard_labels.copy()
y_syn_soft = soft_labels.copy()

In [110]:
pd.Series(y_syn_hard).value_counts()

1    38544
0    37943
Name: count, dtype: int64

In [111]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 3. Downstream Classification Performance

### 3-1. No margin-based filtering

In [112]:
import numpy as np
import copy
from typing import Tuple, Dict, Union, List, Optional

# Assuming these are available in your environment as per your existing code
from softprompt.metrics.evaluator import BinaryEvaluator
from softprompt.algorithms.sklearn.linear_model import (
    SoftLogisticRegressionCV
)
# Standard sklearn LogisticRegressionCV
from sklearn.linear_model import LogisticRegressionCV


def evaluate_logreg_cv_experiment(
    X_train_full: np.ndarray,
    y_train_full_probs: np.ndarray,  # Original training labels as probabilities
    X_test: np.ndarray,
    y_test_hard: np.ndarray,         # Test labels, already hard (0 or 1)
    subsample_size: int,
    model_variant: str,              # 'soft' or 'standard'
    Cs: Union[int, List[float], np.ndarray] = 10,
    cv_folds: int = 5,
    solver: str = 'lbfgs',
    max_iter_solver: int = 1000,
    n_jobs_cv: Optional[int] = 8,
    base_random_seed: int = 42,
    bootstrap: bool = True,
    n_trials: int = 50
) -> Tuple[Dict[str, List[float]], Dict[str, List[float]]]:

    if X_train_full.shape[0] != len(y_train_full_probs):
        raise ValueError("X_train_full and y_train_full_probs must have the same number of samples.")
    if model_variant not in ['soft', 'standard', 'gce']:
        raise ValueError("model_variant must be 'soft' or 'standard'.")

    evaluator = BinaryEvaluator(threshold=0.5) # Assuming BinaryEvaluator is defined elsewhere
    
    # Initialize dictionaries to store metric arrays for each trial
    # The keys will be metric names from evaluator.metrics_to_compute
    # This assumes evaluator.metrics_to_compute is available and populated before this call
    if not hasattr(evaluator, 'metrics_to_compute') or not evaluator.metrics_to_compute:
        # Fallback if metrics_to_compute is not set, use a default or raise error
        # For demonstration, let's assume some common metrics if not available.
        # In a real scenario, BinaryEvaluator should define this.
        print("Warning: evaluator.metrics_to_compute not found or empty. Using default metrics for collection: ['accuracy', 'roc_auc']")
        default_metrics = ['accuracy', 'roc_auc'] # Example
        tr_metric_to_array = {m: np.empty(n_trials) for m in default_metrics}
    else:
        tr_metric_to_array = {m: np.empty(n_trials) for m in evaluator.metrics_to_compute}
    te_metric_to_array = copy.deepcopy(tr_metric_to_array)

    full_idx = np.arange(X_train_full.shape[0])
    
    # Convert full y_train_full_probs to hard labels for evaluation consistency
    y_train_full_hard_for_eval = (y_train_full_probs > 0.5).astype(int)

    for i in range(n_trials):
        current_trial_seed = base_random_seed + i
        rng = np.random.default_rng(current_trial_seed)

        # Get sub-sampled indices to use for training
        if bootstrap:
            subsample_idx = rng.choice(full_idx, size=subsample_size, replace=True)
        else:
            subsample_idx = rng.permutation(full_idx)[:subsample_size]

        X_train_subsample = X_train_full[subsample_idx]
        y_train_subsample_probs = y_train_full_probs[subsample_idx]

        # Fit model
        if model_variant == 'soft':
            lg = SoftLogisticRegressionCV(
                Cs=Cs, cv=cv_folds, solver=solver,
                max_iter=max_iter_solver, n_jobs=n_jobs_cv,
                random_state=current_trial_seed,
            )
            # Prepare P_train for soft labels (num_samples, num_classes)
            # Assuming binary classification where y_train_subsample_probs are probs of class 1
            if y_train_subsample_probs.ndim == 1:
                 P_train_fit = np.stack(
                    [1 - y_train_subsample_probs, y_train_subsample_probs], axis=1
                )
            elif y_train_subsample_probs.ndim == 2 and y_train_subsample_probs.shape[1] == 2:
                 P_train_fit = y_train_subsample_probs # Assume it's already [prob_class_0, prob_class_1]
            else:
                raise ValueError(
                    "y_train_subsample_probs must be 1D (probs for class 1)"
                    " or 2D with shape (n_samples, 2)."
                )

            lg.fit(X_train_subsample, P_train_fit);

        else: # model_variant == 'standard'
            lg = LogisticRegressionCV(
                Cs=Cs, cv=cv_folds, solver=solver,
                max_iter=max_iter_solver, n_jobs=n_jobs_cv,
                random_state=current_trial_seed,
                penalty='l2',  # Assuming L2 penalty to align with SoftLogisticRegression
                scoring='neg_log_loss' # To align CV optimization goal with 'cross_entropy'
            )
            # Standard LogisticRegressionCV expects 1D hard labels for y
            y_train_fit_hard = (y_train_subsample_probs > 0.5).astype(int)
            lg.fit(X_train_subsample, y_train_fit_hard)

        # Evaluate on the full training set
        # The evaluator expects y_true as hard labels and y_prob as predicted probabilities
        tr_metrics_i = evaluator(
            y_true=y_train_full_hard_for_eval, # Hard labels from original full y_train
            y_prob=lg.predict_proba(X_train_full)
        )
        for m, v in tr_metrics_i.items():
            if m in tr_metric_to_array:
                 tr_metric_to_array[m][i] = v

        # Evaluate on the test set
        te_metrics_i = evaluator(
            y_true=y_test_hard, # y_test is already hard labels
            y_prob=lg.predict_proba(X_test)
        )
        for m, v in te_metrics_i.items():
            if m in te_metric_to_array:
                te_metric_to_array[m][i] = v
    
    # Aggregate metrics (train)
    tr_out_agg = {}
    for m, arr in tr_metric_to_array.items():
        if not np.all(np.isnan(arr)): # Ensure array is not all NaNs before mean/std
            tr_out_agg[m] = [np.nanmean(arr), np.nanstd(arr, ddof=1)]
        else:
            tr_out_agg[m] = [np.nan, np.nan]


    # Aggregate metrics (test)
    te_out_agg = {}
    for m, arr in te_metric_to_array.items():
        if not np.all(np.isnan(arr)):
            te_out_agg[m] = [np.nanmean(arr), np.nanstd(arr, ddof=1)]
        else:
            te_out_agg[m] = [np.nan, np.nan]

    return tr_out_agg, te_out_agg

`Standard`

In [ ]:
N = len(X_train) # Changed from X_train to X_syn
ratios = (0.01, 0.05, 0.10, 0.25, 0.50, 1.00)
subsample_sizes = [int(N * r) for r in ratios]

# Select which model variant to run in this loop
# Options: 'soft' or 'standard' or 'gce'
current_model_variant = 'standard'

for i, subsample_size in enumerate(subsample_sizes):

    from rich.console import Console
    console = Console()
    console.print(f">> Model Variant: [bold yellow]{current_model_variant.upper()}[/bold yellow], "
          f"Subsample Ratio: {ratios[i]:.2f} (Size: {subsample_size:,})")

    # (optional) Convert y_test probabilities to hard labels for y_test_hard argument
    y_test_hard_labels = (y_test > 0.5).astype(int)

    tr_metrics, te_metrics = evaluate_logreg_cv_experiment(
        X_train_full=X_syn,
        y_train_full_probs=y_syn_soft,
        X_test=X_test,
        y_test_hard=y_test_hard_labels,
        subsample_size=subsample_size,
        model_variant=current_model_variant, 
        bootstrap=True,
        n_trials=50
    )

    from rich.table import Table
    table = Table(title=f"Metrics (r={ratios[i]:.2f})")
    table.add_column("Metric Name", style='cyan', no_wrap=True, justify='left')
    table.add_column("Train", style='blue', justify='center')
    table.add_column("Test", style='white', justify='center')
    for name in tr_metrics.keys():  # assuming that keys are identical
        tr_mean_, tr_std_ = tr_metrics[name]
        tr_metric_value_str = f"{tr_mean_:.4f} (± {tr_std_:.4f})"
        te_mean_, te_std_ = te_metrics[name]
        te_metric_value_str = f"{te_mean_:.4f} (± {te_std_:.4f})"
        table.add_row(
            name,
            tr_metric_value_str,
            te_metric_value_str,
        )
    console.print(table);

    console.print("-" * 50)

### 3-2. Margin-based Filtering

In [113]:
console = Console()

margin_to_metrics = {}
margins = (0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30)

# Select which model variant to run in this loop
# Options: 'soft' or 'standard' or 'gce'
current_model_variant = 'standard'

for _, m in enumerate(margins):
    console.rule(f"[bold cyan]Margin: {m:.2f}[/bold cyan]")

    # Filter data based on margin using y_syn_soft (1D probabilities)
    margin_mask = \
        (pd.Series(y_syn_soft) <= (0.5 - m)) | \
        (pd.Series(y_syn_soft) >= (0.5 + m))
    effective_indices = np.where(margin_mask.values)[0]

    if len(effective_indices) == 0:
        console.print(f">> Effective pool size is 0 with margin = {m}. Skipping.")
        margin_to_metrics[m] = {'train': {}, 'test': {}} # Store empty results
        continue

    X_syn_filtered = X_syn[effective_indices]
    y_syn_soft_filtered = y_syn_soft[effective_indices] # This is 1D
    N_filtered = len(X_syn_filtered)
    console.print(f">> Effective pool size is {N_filtered:,} with margin = {m}")

    # Store metrics for this margin across different subsample ratios
    train_metrics_for_margin_all_ratios: Dict[str, List[List[float]]] = {}
    test_metrics_for_margin_all_ratios: Dict[str, List[List[float]]] = {}
    
    N = len(X_train)  # 
    ratios = (0.01, 0.05, 0.10, 0.25, 0.50, 1.00)
    subsample_sizes = [int(N * r) for r in ratios]

    for i, subsample_size in enumerate(subsample_sizes):
        
        console.print(f"\t>> Model: [bold yellow]{current_model_variant.upper()}[/bold yellow], "
                      f"Subsample Ratio: {ratios[i]:.2f} (Size: {subsample_size:,})")

        # Convert y_test probabilities to hard labels for y_test_hard argument
        y_test_hard_labels = (y_test > 0.5).astype(int)

        tr_metrics, te_metrics = evaluate_logreg_cv_experiment(
            X_train_full=X_syn_filtered,
            y_train_full_probs=y_syn_soft_filtered, # Pass the filtered 1D soft labels
            X_test=X_test,
            y_test_hard=y_test_hard_labels,
            subsample_size=subsample_size,
            model_variant=current_model_variant,
            bootstrap=True,
            n_trials=50, # Fewer trials for faster example run
        )
        
        # Store all metrics for this ratio
        for metric_name, (mean_val, std_val) in tr_metrics.items():
            if metric_name not in train_metrics_for_margin_all_ratios:
                train_metrics_for_margin_all_ratios[metric_name] = []
            train_metrics_for_margin_all_ratios[metric_name].append([mean_val, std_val])
        
        for metric_name, (mean_val, std_val) in te_metrics.items():
            if metric_name not in test_metrics_for_margin_all_ratios:
                test_metrics_for_margin_all_ratios[metric_name] = []
            test_metrics_for_margin_all_ratios[metric_name].append([mean_val, std_val])

        from rich.table import Table
        table = Table(title=f"Metrics (m={m:.2f}, r={ratios[i]:.2f})")
        table.add_column("Metric Name", style='cyan', no_wrap=True, justify='left')
        table.add_column("Train", style='blue', justify='center')
        table.add_column("Test", style='white', justify='center')
        for name in tr_metrics.keys():
            tr_mean_, tr_std_ = tr_metrics[name]
            tr_metric_value_str = f"{tr_mean_:.4f} (± {tr_std_:.4f})"
            te_mean_, te_std_ = te_metrics[name]
            te_metric_value_str = f"{te_mean_:.4f} (± {te_std_:.4f})"
            table.add_row(
                name,
                tr_metric_value_str,
                te_metric_value_str,
            )
        console.print(table);

    margin_to_metrics[m] = {
        'train': train_metrics_for_margin_all_ratios,
        'test': test_metrics_for_margin_all_ratios,
    }

    console.print("-" * 50)

────────────────────────────────────────────────── Margin: 0.00 ───────────────────────────────────────────────────

>> Effective pool size is 76,487 with margin = 0.0

>> Model: STANDARD, Subsample Ratio: 0.01 (Size: 250)

               Metrics (m=0.00, r=0.01)                
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 0.9892 (± 0.0021) │ 0.7518 (± 0.0475) │
│ precision   │ 0.9925 (± 0.0028) │ 0.6779 (± 0.0541) │
│ recall      │ 0.9860 (± 0.0049) │ 0.9804 (± 0.0287) │
│ f1_score    │ 0.9892 (± 0.0021) │ 0.7995 (± 0.0285) │
│ roc_auc     │ 0.9988 (± 0.0003) │ 0.9311 (± 0.0176) │
│ auprc       │ 0.9985 (± 0.0005) │ 0.9205 (± 0.0212) │
│ ece         │ 0.0070 (± 0.0037) │ 0.2145 (± 0.0642) │
└─────────────┴───────────────────┴───────────────────┘

>> Model: STANDARD, Subsample Ratio: 0.05 (Size: 1,250)

...

>> Model: STANDARD, Subsample Ratio: 0.50 (Size: 12,500)

               Metrics (m=0.30, r=0.50)                
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 1.0000 (± 0.0000) │ 0.9029 (± 0.0027) │
│ precision   │ 1.0000 (± 0.0000) │ 0.9407 (± 0.0041) │
│ recall      │ 0.9999 (± 0.0000) │ 0.8599 (± 0.0100) │
│ f1_score    │ 1.0000 (± 0.0000) │ 0.8985 (± 0.0036) │
│ roc_auc     │ 1.0000 (± 0.0000) │ 0.9713 (± 0.0002) │
│ auprc       │ 1.0000 (± 0.0000) │ 0.9702 (± 0.0001) │
│ ece         │ 0.0018 (± 0.0000) │ 0.0895 (± 0.0030) │
└─────────────┴───────────────────┴───────────────────┘

>> Model: STANDARD, Subsample Ratio: 1.00 (Size: 25,000)

               Metrics (m=0.30, r=1.00)                
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Metric Name ┃       Train       ┃       Test        ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ accuracy    │ 1.0000 (± 0.0000) │ 0.9032 (± 0.0031) │
│ precision   │ 1.0000 (± 0.0000) │ 0.9403 (± 0.0045) │
│ recall      │ 0.9999 (± 0.0000) │ 0.8612 (± 0.0112) │
│ f1_score    │ 1.0000 (± 0.0000) │ 0.8990 (± 0.0041) │
│ roc_auc     │ 1.0000 (± 0.0000) │ 0.9714 (± 0.0002) │
│ auprc       │ 1.0000 (± 0.0000) │ 0.9702 (± 0.0001) │
│ ece         │ 0.0010 (± 0.0001) │ 0.0782 (± 0.0039) │
└─────────────┴───────────────────┴───────────────────┘

--------------------------------------------------